In [3]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import pickle

In [2]:
# load prepared data
data = np.load("citeseer_prepared.npz", allow_pickle=True)

X = data["X"]
y = data["y"]
train_indices = data["train_indices"]
test_indices = data["test_indices"]
node_ids = data["node_ids"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Training nodes:", len(train_indices))
print("Test nodes:", len(test_indices))

X shape: (3312, 3703)
y shape: (3312,)
Training nodes: 2649
Test nodes: 663


In [4]:
# load neighborhoods
with open("citeseer_neighbors.pkl", "rb") as f:
    neighbors = pickle.load(f)

print("Number of nodes:", len(neighbors))

Number of nodes: 3312


In [5]:
# verify one complete example
node_id = node_ids[0]

center_features = X[0]

neighbor_ids = list(neighbors[node_id])

neighbor_features = np.array([
    X[np.where(node_ids == neighbor_id)[0][0]]
    for neighbor_id in neighbor_ids
])

print("Node:", node_id)
print("Center shape:", center_features.shape)
print("Number of neighbors:", len(neighbor_ids))
print("Neighbor shape:", neighbor_features.shape)
print("Label:", y[0])

Node: 100157
Center shape: (3703,)
Number of neighbors: 12
Neighbor shape: (12, 3703)
Label: 1


In [6]:
# recreate the node-to-index lookup
node_to_index = {
    node_id: index
    for index, node_id in enumerate(node_ids)
}

print("Mapped nodes:", len(node_to_index))

Mapped nodes: 3312


In [7]:
# create a helper function
def get_node_data(node_id):
    center_index = node_to_index[node_id]

    center_features = X[center_index]

    neighbor_ids = list(neighbors[node_id])

    neighbor_features = np.array([
        X[node_to_index[neighbor_id]]
        for neighbor_id in neighbor_ids
    ])

    label = y[center_index]

    return center_features, neighbor_features, label

# test the helper function
center, neighbor_features, label = get_node_data(node_ids[0])

print("Center shape:", center.shape)
print("Neighbor shape:", neighbor_features.shape)
print("Label:", label)

Center shape: (3703,)
Neighbor shape: (12, 3703)
Label: 1


In [8]:
# create the citeseer dataset
from torch.utils.data import Dataset


class CiteseerDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        node_index = self.indices[idx]
        node_id = node_ids[node_index]

        center_features, neighbor_features, label = get_node_data(node_id)

        return center_features, neighbor_features, label

In [9]:
# create train-test datasets
train_dataset = CiteseerDataset(train_indices)
test_dataset = CiteseerDataset(test_indices)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Training samples: 2649
Test samples: 663


In [10]:
# test one sample
center, neighbors_sample, label = train_dataset[0]

print("Center shape:", center.shape)
print("Neighbors shape:", neighbors_sample.shape)
print("Label:", label)
print("Center dtype:", center.dtype)
print("Neighbors dtype:", neighbors_sample.dtype)

Center shape: (3703,)
Neighbors shape: (1, 3703)
Label: 0
Center dtype: float32
Neighbors dtype: float32


In [11]:
# create collate function
def citeseer_collate(batch):
    centers = []
    neighbor_features = []
    labels = []

    for center, neighbors, label in batch:
        centers.append(torch.tensor(center, dtype=torch.float32))
        neighbor_features.append(
            torch.tensor(neighbors, dtype=torch.float32)
        )
        labels.append(label)

    centers = torch.stack(centers)
    labels = torch.tensor(labels, dtype=torch.long)

    return centers, neighbor_features, labels

In [12]:
# create a small dataloader
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=citeseer_collate
)

centers, neighbor_features, labels = next(iter(train_loader))

print("Centers shape:", centers.shape)
print("Number of neighbor sets:", len(neighbor_features))
print("Neighbor shapes:")

for i, neighbors in enumerate(neighbor_features):
    print(f"  Sample {i}: {neighbors.shape}")

print("Labels shape:", labels.shape)

Centers shape: torch.Size([4, 3703])
Number of neighbor sets: 4
Neighbor shapes:
  Sample 0: torch.Size([1, 3703])
  Sample 1: torch.Size([2, 3703])
  Sample 2: torch.Size([21, 3703])
  Sample 3: torch.Size([2, 3703])
Labels shape: torch.Size([4])


In [13]:
# verify validity of labels
print("Labels in batch:", labels.tolist())
print("Unique labels:", torch.unique(labels).tolist())

Labels in batch: [5, 2, 5, 5]
Unique labels: [2, 5]
